# Block Relocation Problem: demonstração

Este notebook roda, no Google Colab, o código do repositório
[ufba-relocation-problem-analysis](https://github.com/alanssoares/ufba-relocation-problem-analysis).
Não é preciso instalar nada: o código usa só a biblioteca padrão do Python.

**Como usar:** rode as células na ordem com **Shift + Enter**, ou use *Ambiente de execução → Executar tudo*.
Os campos à direita de cada célula (arquivo, instância, tempo...) podem ser alterados na hora.

| Passo | O que faz |
|---|---|
| 1 | Clona o repositório e carrega o código |
| 2 | Roda os testes unitários |
| 3 | Escolhe uma instância e desenha o pátio |
| 4 | Roda todos os métodos nas duas variantes e compara |
| 5 | Mostra uma solução movimento a movimento |
| 6 | Exemplo: restrito (5 movimentos) × irrestrito (4) |
| 7 | Instância digitada na hora |
| 8 | Comparação completa num arquivo de 10 instâncias |
| 9 | Resultados completos já salvos no repositório |

**Variantes.** Na **restrita** (a do `LEIAME.md`), só os contêineres acima do alvo atual podem ser relocados.
Na **irrestrita** (a do artigo de Jin, Zhu & Lim, 2015), qualquer topo pode ser relocado.

In [ ]:
#@title 1. Preparar o ambiente (clona o repositório e carrega o código)
import os, sys, importlib

REPO = "https://github.com/alanssoares/ufba-relocation-problem-analysis.git"
DIR = "/content/ufba-relocation-problem-analysis"
if not os.path.exists(DIR):
    !git clone -q {REPO} {DIR}
else:
    !git -C {DIR} pull -q
%cd {DIR}

sys.path.insert(0, os.path.join(DIR, "brp_study"))
import demo
importlib.reload(demo)
from demo import list_files, get_instance, instance_from_stacks, render, solve_all, table_md, replay, find
from IPython.display import Markdown, display

!git log --oneline -1
!python --version
print("Arquivos de instâncias:", len(list_files()))

## 2. Testes unitários

Os testes devem rodar em poucos segundos. Eles cobrem:

- as regras do verificador nas duas variantes;
- o A\* e o A\*-U conferidos contra uma busca em largura exaustiva;
- a validade do limite inferior;
- os casos do nível inferior do GLAH;
- a validade das soluções de todos os métodos;
- a integridade dos arquivos de dados.

In [ ]:
#@title 2. Rodar os testes
!python -m unittest discover -s brp_study -v

## 3. Escolher uma instância

O desenho mostra cada pilha com a base embaixo, e o número de cada contêiner é a sua ordem de retirada.
O conjunto A vai de W3 a W5; o conjunto B tem W = 7.

In [ ]:
#@title 3. Escolher uma instância { run: "auto" }
ARQUIVO = "brp_W3_H4_N8.json"  #@param ["brp_W3_H4_N8.json", "brp_W4_H4_N10.json", "brp_W4_H5_N12.json", "brp_W5_H4_N15.json", "brp_W5_H5_N18.json", "brp_W7_H5_N16.json", "brp_W7_H5_N23.json", "brp_W7_H5_N31.json", "brp_W7_H6_N18.json", "brp_W7_H6_N28.json", "brp_W7_H6_N37.json", "brp_W7_H7_N22.json", "brp_W7_H7_N32.json", "brp_W7_H7_N43.json", "brp_W7_H8_N24.json", "brp_W7_H8_N37.json", "brp_W7_H8_N49.json"]
ID = 8  #@param {type:"slider", min:1, max:10, step:1}

inst = get_instance(ARQUIVO, ID)
print(f"{inst.file}  id={inst.inst_id}  W={inst.W}  H={inst.H}  N={inst.N}")
print()
print(render(inst.stacks, inst.H))

## 4. Rodar todos os métodos nesta instância

| Variante | Métodos |
|---|---|
| Restrita | Min-Max, A\* (exato), GLAH-R com D=3 e D=4 |
| Irrestrita | GLAH-U com D=3 e D=4, A\*-U (exato) |

- **Verificação:** toda solução passa pelo verificador independente, com as regras da sua variante.
- **Tempo:** o limite vale para cada método. Nas instâncias grandes do conjunto B, os métodos exatos podem
  esgotar o limite; nesse caso o status é `timeout` e a coluna "Limite inferior" mostra até onde a prova chegou.
- **Atenção:** com o limite no máximo, uma instância grande pode levar alguns minutos para rodar todos os métodos.

In [ ]:
#@title 4. Comparar os métodos
TEMPO_LIMITE = 10  #@param {type:"slider", min:1, max:60, step:1}
VARIANTES = "as duas"  #@param ["as duas", "restrita", "irrestrita"]

escolha = {"as duas": ("restricted", "unrestricted"),
           "restrita": ("restricted",),
           "irrestrita": ("unrestricted",)}[VARIANTES]
rows = solve_all(inst, time_limit=TEMPO_LIMITE, variants=escolha)
display(Markdown(table_md(rows)))

## 5. Ver uma solução passo a passo

Cada linha mostra o contêiner movido, a origem e o destino, e o tipo do movimento:

- **"bloqueia o alvo"**: o contêiner estava acima do alvo atual. É o único tipo permitido na variante restrita.
- **"antecipada"**: o contêiner não estava acima do alvo. Esse tipo só existe na variante irrestrita.

Os contêineres retirados logo depois de cada movimento também aparecem. Escolha um método que tenha
rodado no passo 4.

In [ ]:
#@title 5. Reproduzir a solução de um método
METODO = "A*-U"  #@param ["A*-U", "GLAH-U (D=3)", "GLAH-U (D=4)", "A*", "GLAH-R (D=3)", "GLAH-R (D=4)", "Min-Max"]
MOSTRAR_PATIO = True  #@param {type:"boolean"}

r = find(rows, METODO)
if r["moves"] is None:
    print(f"{METODO} não devolveu solução (status: {r['status']}).")
else:
    print(replay(inst, r["moves"], restricted=(r["variant"] == "restrita"), show_yard=MOSTRAR_PATIO))

## 6. Exemplo: quanto vale uma relocação antecipada

Instância `brp_W3_H4_N8`, id 8. Os dois valores são ótimos provados: 5 movimentos no restrito e 4 no irrestrito.
No irrestrito, os dois primeiros movimentos esvaziam a pilha P3 *antes* de mexer no alvo, e isso cria um destino
bom para o 3 e para o 6. A discussão completa está no `brp_study/ACHADOS.md`, seção 5.3.

In [ ]:
#@title 6. Restrito × irrestrito no exemplo
ex = get_instance("brp_W3_H4_N8.json", 8)
ex_rows = solve_all(ex, time_limit=10)
display(Markdown(table_md(ex_rows)))
print("=" * 60, "\nRESTRITO (A*)\n")
print(replay(ex, find(ex_rows, "A*")["moves"], restricted=True))
print("=" * 60, "\nIRRESTRITO (A*-U)\n")
print(replay(ex, find(ex_rows, "A*-U")["moves"], restricted=False))

## 7. Instância própria

Digite as pilhas da base para o topo, com contêineres numerados de 1 a N sem repetição.
O exemplo é o pátio do `LEIAME.md`. Depois de rodar esta célula, a variável `inst` passa a ser esta
instância, e o **passo 5** mostra as soluções dela.

In [ ]:
#@title 7. Rodar uma instância digitada
PILHAS = "[[8, 7, 6], [2], [1, 3, 4, 5]]"  #@param {type:"string"}
ALTURA_MAXIMA = 4  #@param {type:"integer"}
TEMPO_LIMITE_PROPRIA = 10  #@param {type:"slider", min:1, max:60, step:1}

import ast
inst = instance_from_stacks(ast.literal_eval(PILHAS), ALTURA_MAXIMA)
print(render(inst.stacks, inst.H))
rows = solve_all(inst, time_limit=TEMPO_LIMITE_PROPRIA)
display(Markdown(table_md(rows)))

## 8. Comparação completa num arquivo (10 instâncias)

Roda o mesmo script dos experimentos oficiais (`run_experiments.py`), com todos os métodos e verificação de
todas as soluções, e mostra os resumos. O filtro é um trecho do nome do arquivo, por exemplo `W5_H5` ou `W7_H5_N23`.

Com 5 s de limite, os arquivos do conjunto A rodam em segundos. Os arquivos grandes do conjunto B podem levar
vários minutos.

In [ ]:
#@title 8. Rodar a comparação num arquivo
FILTRO = "W5_H5"  #@param {type:"string"}
TEMPO_POR_METODO = 5  #@param {type:"slider", min:1, max:30, step:1}
VARIANTE = "both"  #@param ["both", "restricted", "unrestricted"]

OUT = "/content/resultados"
!rm -rf {OUT}
!python brp_study/run_experiments.py --variant {VARIANTE} --only "{FILTRO}" --time-limit {TEMPO_POR_METODO} --out-dir {OUT}
for nome in ["summary.md", "summary_unrestricted.md"]:
    caminho = os.path.join(OUT, nome)
    if os.path.exists(caminho):
        display(Markdown(open(caminho, encoding="utf-8").read()))

## 9. Resultados completos já salvos

Estes são os resultados da execução completa: 170 instâncias e 30 s por método, numa máquina local.
Eles estão versionados no repositório. Rodar tudo de novo leva cerca de 50 minutos, por isso a célula só
mostra os resumos salvos.

In [ ]:
#@title 9. Mostrar os resumos salvos (seção 1 do ACHADOS.md e ganho do irrestrito)
achados = open("brp_study/ACHADOS.md", encoding="utf-8").read()
display(Markdown(achados.split("\n---\n")[1]))  # 1. Resumo em números

resumo_u = open("brp_study/results/summary_unrestricted.md", encoding="utf-8").read()
display(Markdown("## " + resumo_u.split("\n## ")[-1]))  # ganho do irrestrito por arquivo